# 03 — Training Results Analysis

After running `uv run python -m weather_ml.train`, this notebook loads checkpoints
and visualizes predictions vs ground truth.

In [ ]:
import sys

sys.path.insert(0, "../src")

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from weather_ml.data.dataset import WeatherBenchDataModule
from weather_ml.models.linear import LinearBaseline
from weather_ml.models.unet import SimpleUNet

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Load data

In [ ]:
dm = WeatherBenchDataModule(data_dir="../data", batch_size=16, num_workers=0)
dm.setup("test")

test_ds = dm.test_ds
variables = test_ds.variables
print(f"Test samples: {len(test_ds)}")
print(f"Variables: {variables}")

## 2. Load a trained model

Update the checkpoint path below to point to your best checkpoint.

In [ ]:
# Find checkpoint files
ckpt_dirs = list(Path("../outputs").rglob("*.ckpt"))
if ckpt_dirs:
    print("Found checkpoints:")
    for p in ckpt_dirs:
        print(f"  {p}")
    ckpt_path = str(ckpt_dirs[0])
else:
    print("No checkpoints found. Run training first!")
    ckpt_path = None

In [ ]:
if ckpt_path:
    # Try loading as LinearBaseline first, then UNet
    try:
        model = LinearBaseline.load_from_checkpoint(ckpt_path)
        model_name = "LinearBaseline"
    except Exception:
        model = SimpleUNet.load_from_checkpoint(ckpt_path)
        model_name = "SimpleUNet"

    model.eval()
    print(f"Loaded {model_name} from {ckpt_path}")

## 3. Predict and visualize

In [ ]:
if ckpt_path:
    # Pick a sample from the test set
    idx = 100
    x, y_true = test_ds[idx]

    with torch.no_grad():
        y_pred = model(x.unsqueeze(0)).squeeze(0)

    # Denormalize for visualization
    mean = torch.from_numpy(test_ds.mean.squeeze(0).astype(np.float32))
    std = torch.from_numpy(test_ds.std.squeeze(0).astype(np.float32))

    x_denorm = x * std + mean
    y_true_denorm = y_true * std + mean
    y_pred_denorm = y_pred * std + mean

    for i, var in enumerate(variables):
        fig, axes = plt.subplots(1, 3, figsize=(18, 4))

        vmin = min(y_true_denorm[i].min(), y_pred_denorm[i].min())
        vmax = max(y_true_denorm[i].max(), y_pred_denorm[i].max())

        axes[0].imshow(
            y_true_denorm[i].numpy(),
            cmap="viridis",
            aspect="auto",
            vmin=vmin,
            vmax=vmax,
        )
        axes[0].set_title(f"Ground Truth: {var}")

        axes[1].imshow(
            y_pred_denorm[i].numpy(),
            cmap="viridis",
            aspect="auto",
            vmin=vmin,
            vmax=vmax,
        )
        axes[1].set_title(f"Prediction: {var}")

        err = (y_pred_denorm[i] - y_true_denorm[i]).numpy()
        im = axes[2].imshow(err, cmap="RdBu_r", aspect="auto")
        axes[2].set_title(f"Error: {var}")
        plt.colorbar(im, ax=axes[2])

        plt.tight_layout()
        plt.show()

## 4. Compute test metrics

In [ ]:
if ckpt_path:
    # RMSE over a subset of test samples
    n_eval = min(500, len(test_ds))
    sq_errors = []

    for i in range(n_eval):
        x, y_true = test_ds[i]
        with torch.no_grad():
            y_pred = model(x.unsqueeze(0)).squeeze(0)
        # Denormalize
        y_true_d = y_true * std + mean
        y_pred_d = y_pred * std + mean
        sq_errors.append(((y_pred_d - y_true_d) ** 2).mean(dim=(1, 2)).numpy())

    rmse = np.sqrt(np.mean(sq_errors, axis=0))
    for i, var in enumerate(variables):
        print(f"{var} RMSE: {rmse[i]:.2f}")

## 5. Persistence baseline comparison

The simplest baseline: predict that T+6h = T (no change).

In [ ]:
persist_sq_errors = []
n_eval = min(500, len(test_ds))

for i in range(n_eval):
    x, y_true = test_ds[i]
    # Denormalize both
    x_d = x * std + mean
    y_true_d = y_true * std + mean
    # Persistence: predict input = output
    persist_sq_errors.append(((x_d - y_true_d) ** 2).mean(dim=(1, 2)).numpy())

persist_rmse = np.sqrt(np.mean(persist_sq_errors, axis=0))
for i, var in enumerate(variables):
    print(f"{var} Persistence RMSE: {persist_rmse[i]:.2f}")

if ckpt_path:
    print()
    for i, var in enumerate(variables):
        improvement = (1 - rmse[i] / persist_rmse[i]) * 100
        print(f"{var} Model improvement over persistence: {improvement:.1f}%")